# cAPTure: graph-pilot structural audit

This notebook runs only Stage 1 of the pilot: it constructs five-second snapshots in memory and audits size, connectivity, multiplicity, temporal change, benign/attack mixing, shortcuts, and resources. It does not train models and does not access Test1, Test2, `train_pub_exf`, or `train_user_prop`.

The recommended sequence is **SMOKE → manual review → FULL_DEV**. Results are descriptive and do not automatically approve the structural gate.

## 1. Mount Drive and prepare the environment

Before opening Colab, push these changes to the branch configured below. This section installs only the existing CPU dependencies; it does not run tests.

In [1]:
from google.colab import drive
drive.mount("/content/drive")

from datetime import datetime, timezone
from pathlib import Path
import json
import subprocess
import sys

REPOSITORY_URL = "https://github.com/tatipar/temporalgnn-nids.git"
REPOSITORY_BRANCH = "feat/capture-feasibility"
PROJECT_ROOT = Path("/content/temporalgnn-nids")
DRIVE_ROOT = Path("/content/drive/MyDrive/capture_gate0")
LOCAL_WORK_ROOT = Path("/content/capture_graph_structural_work")

if not PROJECT_ROOT.exists():
    subprocess.run(
        ["git", "clone", "--branch", REPOSITORY_BRANCH, "--single-branch",
         REPOSITORY_URL, str(PROJECT_ROOT)],
        check=True,
    )

branch = subprocess.check_output(
    ["git", "branch", "--show-current"], cwd=PROJECT_ROOT, text=True
).strip()
if branch != REPOSITORY_BRANCH:
    raise RuntimeError(f"Expected branch {REPOSITORY_BRANCH}, found {branch}.")

required_files = [
    "code/python/requirements-capture.txt",
    "code/python/utils/capture_graph_structural.py",
    "configs/capture_experiment_v1.yaml",
    "configs/capture_packet_schema_v1.yaml",
    "configs/capture_graph_structural_audit_v1.yaml",
]
missing = [name for name in required_files if not (PROJECT_ROOT / name).is_file()]
if missing:
    raise FileNotFoundError(f"Update the Colab repository copy first: {missing}")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r",
     str(PROJECT_ROOT / "code/python/requirements-capture.txt")],
    check=True,
)
sys.path.insert(0, str(PROJECT_ROOT / "code/python"))

import pandas as pd
from IPython.display import display, Markdown
from utils.capture_graph_structural import run_capture_graph_structural_audit

print("Repository commit:", subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True
).strip())
print("CPU structural-audit environment is ready.")

Mounted at /content/drive
Repository commit: 492c3a07561a3bfadee7b2baa8a9f8f9b5b6ca80
CPU structural-audit environment is ready.


## 2. Bind the prepared FULL_DEV artifacts

The audit uses the already verified canonical Parquet files. It processes one scenario at a time and uses `/content` for ephemeral staging. `BATCH_SIZE` is an execution parameter, not a scientific decision; reduce it if Colab runs short of RAM.

In [2]:
MANIFEST_PATH = PROJECT_ROOT / "configs/capture_experiment_v1.yaml"
PACKET_SCHEMA_PATH = PROJECT_ROOT / "configs/capture_packet_schema_v1.yaml"
STRUCTURAL_CONFIG_PATH = (
    PROJECT_ROOT / "configs/capture_graph_structural_audit_v1.yaml"
)
PREPARED_RUN_DIR = (
    DRIVE_ROOT / "prepared_runs/20260917T235058_827743Z_prepare_full_dev"
)
BATCH_SIZE = 250_000

for required_path in (
    MANIFEST_PATH, PACKET_SCHEMA_PATH, STRUCTURAL_CONFIG_PATH, PREPARED_RUN_DIR
):
    if not required_path.exists():
        raise FileNotFoundError(f"Required input is missing: {required_path}")

print("Prepared input:", PREPARED_RUN_DIR)
print("Local staging:", LOCAL_WORK_ROOT)

Prepared input: /content/drive/MyDrive/capture_gate0/prepared_runs/20260917T235058_827743Z_prepare_full_dev
Local staging: /content/capture_graph_structural_work


## 3. Run SMOKE

SMOKE processes `train_empty_conn` and `train_dollar_char`, one from each benign block. To resume an existing run, replace `None` with its exact ID.

In [3]:
RUN_SMOKE = False  # Set False only to reuse already reviewed SMOKE artifacts.
SMOKE_RUN_ID = "20260928T143238_799667Z_graph_structural_smoke"  # Provide the existing ID when RUN_SMOKE is False.
if not RUN_SMOKE and SMOKE_RUN_ID is None:
    raise ValueError("Provide the reviewed SMOKE_RUN_ID when RUN_SMOKE is False.")
if SMOKE_RUN_ID is None:
    SMOKE_RUN_ID = (
        datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
        + "_graph_structural_smoke"
    )
SMOKE_OUTPUT_DIR = DRIVE_ROOT / "graph_pilot_runs" / SMOKE_RUN_ID
print("SMOKE run ID:", SMOKE_RUN_ID)
print("SMOKE output:", SMOKE_OUTPUT_DIR)

if RUN_SMOKE:
    SMOKE_RESULT = run_capture_graph_structural_audit(
        manifest_path=MANIFEST_PATH,
        packet_schema_path=PACKET_SCHEMA_PATH,
        structural_config_path=STRUCTURAL_CONFIG_PATH,
        prepared_run_dir=PREPARED_RUN_DIR,
        output_dir=SMOKE_OUTPUT_DIR,
        local_work_root=LOCAL_WORK_ROOT,
        mode="SMOKE",
        batch_size=BATCH_SIZE,
    )
else:
    required_smoke_outputs = [
        "scenario_summary.csv", "percentile_summary.csv",
        "resource_summary.json", "construction_summary.json",
        "shortcut_audit.json", "review.md", "run_status.json",
    ]
    missing_smoke_outputs = [
        name for name in required_smoke_outputs
        if not (SMOKE_OUTPUT_DIR / name).is_file()
    ]
    if missing_smoke_outputs:
        raise FileNotFoundError(
            f"Existing SMOKE artifacts are incomplete: {missing_smoke_outputs}"
        )
    print("Reusing previously reviewed SMOKE artifacts without rerunning them.")

SMOKE run ID: 20260928T143238_799667Z_graph_structural_smoke
SMOKE output: /content/drive/MyDrive/capture_gate0/graph_pilot_runs/20260928T143238_799667Z_graph_structural_smoke
Reusing previously reviewed SMOKE artifacts without rerunning them.


## 4. Review SMOKE

Review row conservation, plausible sizes, Jaccard values, components, mixed windows, runtime, and RAM. Also open `review.md` and `shortcut_audit.json` in Drive before approving FULL_DEV.

In [ ]:
smoke_scenarios = pd.read_csv(SMOKE_OUTPUT_DIR / "scenario_summary.csv")
smoke_percentiles = pd.read_csv(SMOKE_OUTPUT_DIR / "percentile_summary.csv")
smoke_resources = json.loads(
    (SMOKE_OUTPUT_DIR / "resource_summary.json").read_text()
)
smoke_construction = json.loads(
    (SMOKE_OUTPUT_DIR / "construction_summary.json").read_text()
)

display(smoke_scenarios)
display(smoke_percentiles.loc[
    smoke_percentiles["metric"].isin([
        "nodes", "edges", "weak_components",
        "largest_weak_component_fraction", "mean_multiplicity",
        "adjacent_node_jaccard", "adjacent_pair_jaccard",
    ])
].set_index(["scenario", "metric", "percentile"]))
print("Row conservation:", smoke_construction["row_conservation_passed"])
print(json.dumps(smoke_resources, indent=2)[:12000])
display(Markdown((SMOKE_OUTPUT_DIR / "review.md").read_text()))

## 5. Manual approval for FULL_DEV

Set `APPROVE_FULL_DEV` to `True` only after reviewing SMOKE. The note remains visible in the session; document the final scientific gate in the FULL_DEV `review.md`.

In [5]:
APPROVE_FULL_DEV = True
SMOKE_REVIEW_NOTE = (
    "Row conservation passed exactly and construction costs were feasible. "
    "Topology was not constant: both scenarios contained thousands of distinct "
    "endpoint sets and topologies, with median adjacent-pair Jaccard near 0.69 "
    "and a largest component covering 75% to 83% of nodes. FULL_DEV is approved "
    "only to complete the descriptive structural audit. Multiplicity, branching, "
    "components, step differences, and endpoint/pair/protocol shortcuts must be "
    "reviewed before model training."
)

if not APPROVE_FULL_DEV:
    raise RuntimeError(
        "Review the SMOKE artifacts, then set APPROVE_FULL_DEV=True."
    )
if not SMOKE_REVIEW_NOTE.strip():
    raise ValueError("Record a non-empty SMOKE_REVIEW_NOTE before FULL_DEV.")
print("FULL_DEV approved from this reviewed SMOKE:", SMOKE_RUN_ID)

FULL_DEV approved from this reviewed SMOKE: 20260928T143238_799667Z_graph_structural_smoke


## 6. Run FULL_DEV

FULL_DEV processes only the five development scenarios. Each completed scenario is immutable and verifiable; use the same `FULL_DEV_RUN_ID` to resume.

In [6]:
FULL_DEV_RUN_ID = None  # Paste an existing ID here to resume.
if FULL_DEV_RUN_ID is None:
    FULL_DEV_RUN_ID = (
        datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
        + "_graph_structural_full_dev"
    )
FULL_DEV_OUTPUT_DIR = DRIVE_ROOT / "graph_pilot_runs" / FULL_DEV_RUN_ID
print("FULL_DEV run ID:", FULL_DEV_RUN_ID)
print("FULL_DEV output:", FULL_DEV_OUTPUT_DIR)

FULL_DEV_RESULT = run_capture_graph_structural_audit(
    manifest_path=MANIFEST_PATH,
    packet_schema_path=PACKET_SCHEMA_PATH,
    structural_config_path=STRUCTURAL_CONFIG_PATH,
    prepared_run_dir=PREPARED_RUN_DIR,
    output_dir=FULL_DEV_OUTPUT_DIR,
    local_work_root=LOCAL_WORK_ROOT,
    mode="FULL_DEV",
    batch_size=BATCH_SIZE,
)

FULL_DEV run ID: 20260928T145715_010326Z_graph_structural_full_dev
FULL_DEV output: /content/drive/MyDrive/capture_gate0/graph_pilot_runs/20260928T145715_010326Z_graph_structural_full_dev
Staging and auditing train_empty_conn...
train_empty_conn: 5,000 nonempty windows, 815,365 packets
Completed train_empty_conn: 9,360 windows, 1,175,779 packets
Staging and auditing train_qos_mid...
train_qos_mid: 5,000 nonempty windows, 825,890 packets
Completed train_qos_mid: 9,360 windows, 1,499,717 packets
Staging and auditing train_dollar_char...
train_dollar_char: 5,000 nonempty windows, 1,905,832 packets
Completed train_dollar_char: 9,116 windows, 2,882,555 packets
Staging and auditing train_slash_char...
train_slash_char: 5,000 nonempty windows, 3,891,091 packets
Completed train_slash_char: 9,116 windows, 6,425,516 packets
Staging and auditing train_sub_exf...
train_sub_exf: 5,000 nonempty windows, 1,585,745 packets
Completed train_sub_exf: 9,115 windows, 2,225,807 packets
Structural audit comp

## 7. Review FULL_DEV and complete the structural gate

Step tables are descriptive: a window containing multiple steps appears in multiple groups. Do not interpret windows as independent observations. Record the final decision in the `review.md` copy without consulting Test1 or Test2.

In [7]:
qscenario_summary = pd.read_csv(FULL_DEV_OUTPUT_DIR / "scenario_summary.csv")
percentile_summary = pd.read_csv(FULL_DEV_OUTPUT_DIR / "percentile_summary.csv")
step_summary = pd.read_csv(FULL_DEV_OUTPUT_DIR / "step_topology_summary.csv")
shortcut_audit = json.loads(
    (FULL_DEV_OUTPUT_DIR / "shortcut_audit.json").read_text()
)
resource_summary = json.loads(
    (FULL_DEV_OUTPUT_DIR / "resource_summary.json").read_text()
)

display(scenario_summary.set_index("scenario"))
display(percentile_summary.loc[
    percentile_summary["metric"].isin([
        "nodes", "edges", "weak_components",
        "largest_weak_component_fraction", "mean_multiplicity",
        "branching_node_fraction", "adjacent_node_jaccard",
        "adjacent_pair_jaccard",
    ])
].set_index(["scenario", "metric", "percentile"]))
display(step_summary.head(200))
print("Fold endpoint/pair memorisation diagnostics:")
print(json.dumps(shortcut_audit["fold_memorisation_rules"], indent=2)[:20000])
print("Resource summary:")
print(json.dumps(resource_summary, indent=2))
display(Markdown((FULL_DEV_OUTPUT_DIR / "review.md").read_text()))
print("Review file:", FULL_DEV_OUTPUT_DIR / "review.md")

,nonempty_windows,benign_only_windows,mixed_windows,attack_only_windows,mixed_window_fraction,distinct_endpoint_sets,distinct_topologies,most_common_topology_fraction,median_nodes,median_edges,median_adjacent_node_jaccard,median_adjacent_pair_jaccard,median_largest_component_fraction
scenario,,,,,,,,,,,,,
train_empty_conn,9360,4068,5290,2,0.565171,5622,6084,0.023077,13.0,79.0,0.714286,0.680000,0.750000
train_qos_mid,9360,4404,4954,2,0.529274,5140,5614,0.029167,13.0,99.0,0.733333,0.703704,0.750000
train_dollar_char,9116,5569,3545,2,0.388877,4307,4743,0.107394,13.0,189.0,0.720000,0.695652,0.833333
train_slash_char,9116,6386,2728,2,0.299254,3990,4386,0.107284,13.0,176.0,0.714286,0.692308,0.833333
train_sub_exf,9115,6431,2683,1,0.294350,3921,4302,0.107735,13.0,165.0,0.692308,0.678571,0.833333


value  observations
scenario         metric                percentile                         
train_empty_conn nodes                 0            2.000000          9360
                                       1            8.000000          9360
                                       5            9.000000          9360
                                       25          11.000000          9360
                                       50          13.000000          9360
...                                                      ...           ...
train_sub_exf    adjacent_pair_jaccard 50           0.678571          8869
                                       75           0.793103          8869
                                       95           1.000000          8869
                                       99           1.000000          8869
                                       100          1.000000          8869

[360 rows x 2 columns]

,scenario,step,window_type,windows,nodes_mean,nodes_median,edges_mean,edges_median,weak_components_mean,weak_components_median,largest_weak_component_fraction_mean,largest_weak_component_fraction_median,simple_directed_density_mean,simple_directed_density_median,mean_multiplicity_mean,mean_multiplicity_median,branching_node_fraction_mean,branching_node_fraction_median
0,train_dollar_char,__BENIGN__,benign_only,5569,10.316753,12.0,169.540672,148.0,1.652900,2.0,0.881863,0.857143,0.220280,0.147436,8.597356,7.000000,0.079572,0.076923
1,train_dollar_char,brute_force_malformed,mixed,154,15.344156,15.0,311.870130,205.5,2.487013,2.0,0.717701,0.733333,0.112687,0.113961,13.324874,8.704545,0.140929,0.142857
2,train_dollar_char,dollar_char,attack_only,2,4.500000,4.5,419.000000,419.0,1.000000,1.0,1.000000,1.000000,0.383333,0.383333,72.085714,72.085714,0.550000,0.550000
3,train_dollar_char,dollar_char,mixed,2004,16.006986,16.0,579.165170,561.0,2.222056,2.0,0.673582,0.687500,0.111938,0.108333,23.443260,22.000000,0.231020,0.235294
4,train_dollar_char,mqtt_cat,mixed,14,14.428571,15.0,271.000000,238.5,2.142857,2.0,0.786505,0.733333,0.120192,0.121795,11.511027,10.301282,0.183050,0.183333
5,train_dollar_char,nmap_10_T5,mixed,96,21.375000,17.0,4235.562500,239.5,2.500000,2.0,0.707074,0.705882,0.090339,0.095711,68.484818,9.889037,0.122067,0.125000
6,train_dollar_char,nmap_banner,mixed,184,19.402174,15.5,290.543478,225.0,2.353261,2.0,0.716855,0.716518,0.101848,0.108333,9.815199,7.532738,0.143656,0.142857
7,train_dollar_char,nmap_mqtt,mixed,1011,17.308605,17.0,232.739862,188.0,2.913947,3.0,0.647501,0.642857,0.093577,0.091667,9.200755,7.380952,0.163620,0.153846
8,train_dollar_char,nmap_sub,mixed,75,19.200000,16.0,374.280000,221.0,2.293333,2.0,0.750043,0.750000,0.100641,0.104762,11.688204,8.545455,0.145230,0.133333
9,train_dollar_char,scp_inst,mixed,7,15.142857,15.0,233.000000,208.0,2.857143,3.0,0.692249,0.666667,0.106519,0.104762,10.384047,9.650000,0.123735,0.133333


Fold endpoint/pair memorisation diagnostics:
{
  "A": {
    "train_scenarios": [
      "train_empty_conn",
      "train_qos_mid"
    ],
    "validation_scenarios": {
      "train_dollar_char": {
        "endpoint_seen_with_attack_in_fold_train": {
          "tp": 32343,
          "fp": 17057,
          "tn": 1687946,
          "fn": 1145209,
          "precision": 0.6547165991902834,
          "recall": 0.02746630297430602,
          "false_positive_rate": 0.010004087969346681
        },
        "endpoint_attack_only_in_fold_train": {
          "tp": 276,
          "fp": 0,
          "tn": 1705003,
          "fn": 1177276,
          "precision": 1.0,
          "recall": 0.0002343845537182222,
          "false_positive_rate": 0.0
        },
        "directed_pair_seen_with_attack_in_fold_train": {
          "tp": 276,
          "fp": 0,
          "tn": 1705003,
          "fn": 1177276,
          "precision": 1.0,
          "recall": 0.0002343845537182222,
          "false_positive_rate"

# cAPTure structural-audit review

Mode: `FULL_DEV`.

This file is a descriptive template. It does not automatically approve the gate.

## Main signals

| Scenario | Nonempty windows | Median nodes | Median edges | Adjacent-node Jaccard | Dominant-topology fraction | Mixed-window fraction |
|---|---:|---:|---:|---:|---:|---:|
| `train_empty_conn` | 9,360 | 13.00 | 79.00 | 0.7143 | 0.0231 | 0.5652 |
| `train_qos_mid` | 9,360 | 13.00 | 99.00 | 0.7333 | 0.0292 | 0.5293 |
| `train_dollar_char` | 9,116 | 13.00 | 189.00 | 0.7200 | 0.1074 | 0.3889 |
| `train_slash_char` | 9,116 | 13.00 | 176.00 | 0.7143 | 0.1073 | 0.2993 |
| `train_sub_exf` | 9,115 | 13.00 | 165.00 | 0.6923 | 0.1077 | 0.2943 |

## Resources

| Scenario | Total time (s) | Packets/s | Sampled peak RSS (GiB) | Estimated model tensors (GiB) |
|---|---:|---:|---:|---:|
| `train_empty_conn` | 40.89 | 28757.96 | 0.444 | 0.479 |
| `train_qos_mid` | 40.60 | 36941.01 | 0.499 | 0.611 |
| `train_dollar_char` | 40.10 | 71880.06 | 0.478 | 1.174 |
| `train_slash_char` | 45.76 | 140417.01 | 0.516 | 2.616 |
| `train_sub_exf` | 41.86 | 53171.58 | 0.493 | 0.907 |

## Manual review questions

- [ ] Do nodes and pairs change materially between adjacent windows?
- [ ] Do components contain shared neighborhoods, or do isolated pairs dominate?
- [ ] Do step-level differences survive scenario and protocol breakdowns?
- [ ] Do hashed endpoint/pair or protocol rules explain much of the attack traffic?
- [ ] Does construction fit the time, RAM, and storage budgets?
- [ ] Is the one-seed model comparison approved?

Do not access Test1 or Test2 to resolve this review.


Review file: /content/drive/MyDrive/capture_gate0/graph_pilot_runs/20260928T145715_010326Z_graph_structural_full_dev/review.md


## 8. Interpretation boundary

This audit can reveal variation, triviality, shortcuts, and resource feasibility. It does not show that a GNN uses topology. Implement the next stage only after reviewing these artifacts and freezing the memory policy, inner-checkpoint protocol, hyperparameters, and minimum effect sizes.

In [11]:
from datetime import datetime, timezone
import hashlib

SOURCE_REVIEW_PATH = FULL_DEV_OUTPUT_DIR / "review.md"
DECISION_PATH = FULL_DEV_OUTPUT_DIR / "stage1_decision.md"

source_review_sha256 = hashlib.sha256(
    SOURCE_REVIEW_PATH.read_bytes()
).hexdigest()

decision_text = f"""# cAPTure graph pilot: Stage-1 decision

Created: {datetime.now(timezone.utc).isoformat()}
Source run: `{FULL_DEV_RUN_ID}`
Source review SHA-256: `{source_review_sha256}`

## Decision

Status: **PASS_WITH_LIMITATIONS**.

The development scenarios contain genuine but moderately persistent relational
variation, connected neighborhoods, and step-associated structural changes.
Graph construction is feasible, and fold-trained endpoint/pair memorization
does not provide meaningful attack coverage.

Protocol composition is a major interpretation risk. MQTT absence provides
strong single-feature ranking, while broadcast destinations identify a small
but extremely attack-enriched packet subset. Because broadcast and multicast
addresses are also explicit graph nodes, apparent graph gains may reflect
protocol/group-address patterns or edge multiplicity rather than general
relational learning.

Proceed to the one-seed model comparison, retaining identical packet features
across variants and requiring MQTT, broadcast, protocol, scenario, and step
breakdowns. Topology should advance beyond the pilot only if graph models
improve timely coverage or lead time at the same false-alert budget outside
these shortcut strata.

## Gate results

| Gate | Result |
|---|---|
| Data integrity and row conservation | PASS |
| Topological variation | PASS |
| Non-trivial neighborhoods | PASS |
| Construction resources | PASS |
| Endpoint/pair shortcut risk | LOW |
| Protocol shortcut risk | HIGH |
| Approval for one-seed comparison | PASS WITH LIMITATIONS |
| Current evidence that a GNN is necessary | NOT ESTABLISHED |
"""

if DECISION_PATH.exists():
    raise FileExistsError(
        f"{DECISION_PATH} already exists. Review it instead of overwriting it."
    )

DECISION_PATH.write_text(decision_text, encoding="utf-8")

print("Stage-1 decision saved to:")
print(DECISION_PATH)


Stage-1 decision saved to:
/content/drive/MyDrive/capture_gate0/graph_pilot_runs/20260928T145715_010326Z_graph_structural_full_dev/stage1_decision.md
